# Ch 5　Graph API：StateGraph、節點與條件邊

> 我們把 Ch4 的骨架升級成**真的會呼叫 LLM** 的 email agent。
>
> 為了讓沒有 API key 的人也能跑，本 notebook 提供兩種 `classify`：**(離線) 關鍵字版** 與 **(線上) LLM 版**。其餘圖的結構完全相同。

In [7]:
!uv pip install -q langgraph langchain-google-genai grandalf

In [17]:
!uv pip install -q grandalf

In [1]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
# 設定 state 的結構, 包含 email, category, reply, 繼承 TypedDict 
class State(TypedDict):
    email: str
    category: str
    reply: str

### 節點（LLM 版）：需要 GOOGLE_API_KEY

In [2]:
import os, getpass
# 想跑「真 LLM 版」就執行這格設金鑰；只想看結構/路由可跳過，用下一格的離線版。
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("GOOGLE_API_KEY（可留空跳過）：")
# 下面是 LLM 的實作, 用 ChatGoogleGenerativeAI 呼叫 Google 的 LLM
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

# 下面三個函數是節點的實作, 分別是分類和草擬回覆
def classify_llm(state: State) -> dict:
    prompt = ("把以下客服信件分類，只回一個詞：simple 或 complex。\n\n"
              f"信件：{state['email']}")
    out = llm.invoke(prompt).content.strip().lower()
    return {"category": "complex" if "complex" in out else "simple"}

def draft_reply(state: State) -> dict:
    prompt = f"用友善專業的中文語氣，草擬一封回覆這封信的 email：\n\n{state['email']}"
    return {"reply": llm.invoke(prompt).content}

def escalate(state: State) -> dict:
    return {"reply": "（此案已升級給真人客服專員處理。）"}

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


### 節點（離線版）：不需要金鑰，照樣能示範路由

In [3]:
# 離線版利用 rule-based 的簡單邏輯，不需 LLM 金鑰, 因為這裡的邏輯是固定的

def classify_offline(state: State) -> dict:
    complex_kw = any(k in state["email"] for k in ["壞", "當機", "504", "兩次", "complex"])
    return {"category": "complex" if complex_kw else "simple"}

def draft_offline(state: State) -> dict:
    return {"reply": f"[離線草稿] 已收到您的來信：{state['email'][:20]}…我們會盡快協助您。"}

def escalate(state: State) -> dict:                # escalate 不需要 LLM，這裡定義一份方便離線跑
    return {"reply": "（此案已升級給真人客服專員處理。）"}

### 連邊、編譯、執行

把 `USE_LLM` 設成 True/False 切換線上/離線。圖的結構一模一樣——這正是重點：**換掉節點實作，不動圖的骨架。**

In [4]:
# 這邊設定一個環境變數用來切換線上/離線

USE_LLM = False   # 沒金鑰就維持 False；有金鑰可改 True 跑真模型

classify = classify_llm if USE_LLM else classify_offline
draft    = draft_reply  if USE_LLM else draft_offline

def route(state: State) -> str:
    # 路由函數：只讀 state、回一個字串。複雜 -> escalate，否則 -> draft。
    return "draft" if state["category"] == "simple" else "escalate"

b = StateGraph(State)
b.add_node("classify", classify)
b.add_node("draft", draft)
b.add_node("escalate", escalate)
b.add_edge(START, "classify")
b.add_conditional_edges("classify", route, {"draft": "draft", "escalate": "escalate"})
b.add_edge("draft", END)
b.add_edge("escalate", END)
graph = b.compile()

In [6]:
# invoke：拿最終 state
print(graph.invoke({"email": "請問如何重設密碼？", "category": "", "reply": ""}))

# stream(updates)：看它「一步步」走過哪些節點——學習與除錯的利器。
print("\n--- 複雜信件的執行路徑 ---")
for step in graph.stream(
    {"email": "你們的 API 整合會間歇性回 504，服務一直中斷", "category": "", "reply": ""},
    stream_mode="updates",
):
    print(step)
# 你會看到 classify -> escalate；draft 完全沒被呼叫（Ch7 會用執行模型解釋為什麼）。

{'email': '請問如何重設密碼？', 'category': 'simple', 'reply': '[離線草稿] 已收到您的來信：請問如何重設密碼？…我們會盡快協助您。'}

--- 複雜信件的執行路徑 ---
{'classify': {'category': 'complex'}}
{'escalate': {'reply': '（此案已升級給真人客服專員處理。）'}}


## agent loop 在 Graph API 裡長什麼樣？

就是「一條從 tools 節點繞回 model 節點的邊」：
```
START → model → (要用工具?) → tools → model → … → END
```
`create_agent`（Ch2 的 (A)）底下就是這個結構。你現在不只能用它，還能親手在中間插節點、加分支。

## 拓樸圖：眼見為憑

In [9]:
print(graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	classify(classify)
	draft(draft)
	escalate(escalate)
	__end__([<p>__end__</p>]):::last
	__start__ --> classify;
	classify -.-> draft;
	classify -.-> escalate;
	draft --> __end__;
	escalate --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [7]:
print("--- ASCII ---")
print(graph.get_graph().draw_ascii())
print("\n--- Mermaid 原始碼 ---")
print(graph.get_graph().draw_mermaid())

--- ASCII ---
        +-----------+           
        | __start__ |           
        +-----------+           
              *                 
              *                 
              *                 
        +----------+            
        | classify |            
        +----------+            
         ..        .            
       ..           ..          
      .               .         
+-------+         +----------+  
| draft |         | escalate |  
+-------+         +----------+  
         **        *            
           **    **             
             *  *               
         +---------+            
         | __end__ |            
         +---------+            

--- Mermaid 原始碼 ---
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	classify(classify)
	draft(draft)
	escalate(escalate)
	__end__([<p>__end__</p>]):::last
	__start__ --> classify;
	classify -.-> draft;
	classify -.-> escalate;
	draft --> __end